In [1]:
from pathlib import Path
from tqdm import tqdm

import json
import re
import uuid

In [2]:
BASE_DIR = Path("..")

INPUT_DIR = BASE_DIR / "cleaned_docs"

OUTPUT_DIR = BASE_DIR / "chunks"

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
md_files = list(INPUT_DIR.rglob("*.md"))

print(f"Found {len(md_files)} markdown files")

Found 13 markdown files


In [4]:
HEADING_PATTERN = re.compile(r"^(#{1,6})\s+(.*)")

In [5]:
def split_into_sections(markdown_text):

    sections = []

    current_heading = "Introduction"

    current_content = []

    current_level = 0

    for line in markdown_text.splitlines():

        match = HEADING_PATTERN.match(line)

        if match:

            if current_content:

                sections.append({

                    "heading": current_heading,

                    "level": current_level,

                    "content": "\n".join(current_content).strip()

                })

            current_heading = match.group(2).strip()

            current_level = len(match.group(1))

            current_content = []

        else:

            current_content.append(line)

    if current_content:

        sections.append({

            "heading": current_heading,

            "level": current_level,

            "content": "\n".join(current_content).strip()

        })

    return sections

In [6]:
def chunk_text(text,
               chunk_size=350,
               overlap=50):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [7]:
all_chunks = []

for md_file in tqdm(md_files):

    markdown = md_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    sections = split_into_sections(markdown)

    source = md_file.stem

    category = md_file.parent.name

    for section in sections:

        heading = section["heading"]

        level = section["level"]

        content = section["content"]

        pieces = chunk_text(content)

        for index, piece in enumerate(pieces):

            if len(piece.strip()) < 100:
                continue

            all_chunks.append({

                "id": str(uuid.uuid4()),

                "source": source,

                "category": category,

                "heading": heading,

                "heading_level": level,

                "chunk_number": index,

                "text": piece

            })

100%|█████████████████████████████████████████| 13/13 [00:00<00:00, 365.77it/s]


In [8]:
print("Total Chunks :", len(all_chunks))

print()

print(all_chunks[0])

Total Chunks : 2525

{'id': '22e56a99-e8b7-4794-8efc-dfba82a6a164', 'source': 'ndmp-2019', 'category': 'cleaned_docs', 'heading': 'A publication of:', 'heading_level': 2, 'chunk_number': 0, 'text': 'National Disaster Management Authority Government of India NDMA Bhawan A-1, Safdarjung Enclave New Delhi - 110 029 Ministry of Home Affairs Revised Edition - November, 2019 National Disaster Management Plan - 2016 When citing this report the following citation should be used: National Disaster Management Plan, 2019. A publication of the National Disaster Management Authority, Government of India. November 2019, New Delhi.'}


In [9]:
output_file = OUTPUT_DIR / "chunks.json"

with open(output_file,
          "w",
          encoding="utf-8") as f:

    json.dump(
        all_chunks,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved")

print(output_file)

Saved
../chunks/chunks.json


In [10]:
import random

sample = random.choice(all_chunks)

print("="*80)

print("SOURCE :", sample["source"])

print("CATEGORY :", sample["category"])

print("HEADING :", sample["heading"])

print("="*80)

print(sample["text"])

SOURCE : 61_245057_Cyclone Warning SOP Booklet final
CATEGORY : cleaned_docs
HEADING : 5.4.8.1. Nomograms
Ghosh model nomograms are based on the numerical solution to the hydrodynamical equations governing motion of the Sea. The nomograms are prepared relating peak surge with various parameters such as pressure drop, radius of maximum wind, vector motion of the cyclone and offshore bathymetry.
